# Retrieval baseline 

Studying the Income Tax

Imports

In [92]:
import re
import json
import faiss
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [93]:
random.seed(42)

Getting the dataset

In [51]:
questions_dataset = load_dataset(
    "unicamp-dl/rag-rfb",
    data_files="questions_QA_2024_v1.1.json",
    split='train'
)
# questions_dataset = questions_dataset['train']
questions = questions_dataset.to_list()

In [54]:
questions[0].keys()

dict_keys(['question_number', 'question_summary', 'question_text', 'answer', 'answer_cleaned', 'references', 'linked_questions', 'formatted_references', 'embedded_references', 'formatted_embedded_references', 'all_formatted_references'])

In [50]:
documents_dataset = load_dataset(
    "unicamp-dl/rag-rfb",
    data_files="referred_legal_documents_QA_2024_v1.1.json",
    split='train'
)
# documents_dataset = documents_dataset['train']
documents = documents_dataset.to_list()

In [53]:
documents[0]

{'filename': 'ADI RFB nº 12, de 2016.txt',
 'filedata': 'NORMAS Contraste \ue88a \ue8ad Ato Declaratório Interpretativo RFB nº 12, de 23 de novembro de 2016 (Publicado(a) no DOU de 25/11/2016, seção 1, página 27) Multivigente Vigente Original Relacional Dispõe sobre a isenção de Imposto sobre a Renda nas aplicações em Certificado de Direitos Creditórios do Agronegócio e Certificado de Recebíveis do Agronegócio. O SECRETÁRIO DA RECEITA FEDERAL DO BRASIL, no uso das atribuições que lhe conferem os incisos III e XXVI do art. 280 do Regimento Interno da Secretaria da Receita Federal do Brasil, aprovado pela Portaria MF n 203, de 14 de maio de 2012, e tendo em vista o disposto no art. 3 da Lei n 11.033, de 21 de dezembro de 2004, e no art. 37 da Lei n 11.076, de 30 de dezembro de 2004, declara: Art. 1 Enquadra-se no conceito de remuneração para fins da isenção prevista no inciso IV do art. 3 da Lei n 11.033, de 21 de dezembro de 2004, a parcela da variação cambial paga pelo Certificado de D

## Checking

In [37]:
documents[0]

{'filename': 'ADI RFB nº 12, de 2016.txt',
 'filedata': 'NORMAS Contraste \ue88a \ue8ad Ato Declaratório Interpretativo RFB nº 12, de 23 de novembro de 2016 (Publicado(a) no DOU de 25/11/2016, seção 1, página 27) Multivigente Vigente Original Relacional Dispõe sobre a isenção de Imposto sobre a Renda nas aplicações em Certificado de Direitos Creditórios do Agronegócio e Certificado de Recebíveis do Agronegócio. O SECRETÁRIO DA RECEITA FEDERAL DO BRASIL, no uso das atribuições que lhe conferem os incisos III e XXVI do art. 280 do Regimento Interno da Secretaria da Receita Federal do Brasil, aprovado pela Portaria MF n 203, de 14 de maio de 2012, e tendo em vista o disposto no art. 3 da Lei n 11.033, de 21 de dezembro de 2004, e no art. 37 da Lei n 11.076, de 30 de dezembro de 2004, declara: Art. 1 Enquadra-se no conceito de remuneração para fins da isenção prevista no inciso IV do art. 3 da Lei n 11.033, de 21 de dezembro de 2004, a parcela da variação cambial paga pelo Certificado de D

In [42]:
questions_dataset[0]

{'question_number': '001',
 'question_summary': 'OBRIGATORIEDADE',
 'question_text': 'Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?',
 'answer': ['Está obrigada a apresentar a Declaração de Ajuste Anual (DAA) referente ao exercício de 2024, a pessoa',
  'física residente no Brasil que, no ano-calendário de 2023:',
  '1 - recebeu rendimentos tributáveis, sujeitos ao ajuste na declaração, cuja soma foi superior a R$ 30.639,90',
  '(trinta mil, seiscentos e trinta e nove reais e noventa centavos);',
  '2 - recebeu rendimentos isentos, não tributáveis ou tributados exclusivamente na fonte, cuja soma foi superior',
  'a R$ 200.000,00 (duzentos mil reais);',
  '3 - obteve, em qualquer mês, ganho de capital na alienação de bens ou direitos sujeito à incidência do imposto;',
  '4 - realizou operações de alienação em bolsas de valores, de mercadorias, de futuros e assemelhadas:',
  'a) cuja soma foi superior a R$ 40.000,00 (q

## Chunking

In [72]:
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

In [69]:
SEPARATORS = [
    r"\n\s*Art\.\s+\d+[A-Za-zº°-]*",
    r"\n\s*§\s*\d+[A-Za-zº°-]*",
    r"\n\s*Parágrafo único",
    r"\n\s*Inciso\s+[IVXLCDM]+",
    r"\n\s*\n",
    r"\n",
    r"\.\s+",
    r";\s+",
    r",\s+",
    r"\s+",
]

In [70]:
def clean_legal_text(text: str) -> str:
    text = re.sub(r"[\ue000-\uf8ff]", " ", text)  # private-use glyphs
    text = re.sub(r"\*Este texto não substitui o publicado oficialmente\.", " ", text, flags=re.I)
    text = re.sub(r"A visualização deste sistema.*$", " ", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [73]:
def chunking(
    texts: list,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: list = SEPARATORS,
) -> list:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
        is_separator_regex=True,
        add_start_index=True,
    )

    chunks = []
    for i, item in enumerate(tqdm(texts, desc="CHUNKING TEXTS", ncols=100)):
        raw_text = item.get("filedata", "")
        if not raw_text:
            continue

        text = clean_legal_text(raw_text)
        filename = item.get("filename", f"doc_{i}")

        docs = text_splitter.create_documents(
            [text],
            metadatas=[{"doc_id": i, "filename": filename}]
        )

        for k, doc in enumerate(docs):
            chunks.append({
                "doc_id": i,
                "chunk_id": k,
                "filename": filename,
                "start_index": doc.metadata.get("start_index"),
                "text": doc.page_content,
            })

    return chunks

Checking if it's working

In [94]:
checking_chunks = chunking(documents[:100])

CHUNKING TEXTS: 100%|████████████████████████████████████████████| 100/100 [00:00<00:00, 857.71it/s]


In [97]:
checking_chunks[random.randint(0, 200)]

{'doc_id': 1,
 'chunk_id': 3,
 'filename': 'Acordo para Evitar a Dupla Tributação em Matéria de Impostos sobre a Renda e o Capital firmado entre o Brasil e a Alemanha.txt',
 'start_index': 1420,
 'text': 'Regularização de Impostos Consultar dívidas e pendências Pagar impostos Alterar pagamentos Consultar pagamentos Parcelar dívidas Consultar parcelamentos Fazer acordo de transação Revisar débitos e pendências Restituições e Compensações Consultar restituição Obter restituição Compensar impostos Conveniados e Parceiros Estados e Municípios Rede Arrecadadora Casa da Moeda Outras Entidades Assuntos Notícias Todas as notícias Arrecadação e Cobrança Cidadania Fiscal Combate ao contrabando Combate à corrupção Combate à sonegação Institucional Serviços Tributação Agenda Tributária Taxas de Juros Aduana e Comércio Exterior Atendimento via e-CAC Classificação Fiscal de Mercadorias Controle de Carga e'}

## Building Embeddings for PT-BR

Loading embedding model optimized for multilingual/Portuguese content

In [80]:
BATCH_SIZE = 32

In [78]:
models_available = {
    '1': 'all-MiniLM-L6-v2', # Good balance of speed and quality
    '2': 'neuralmind/bert-base-portuguese-cased', # Portuguese-specific
    '3': 'distilbert-base-uncased', # Lightweight and fast
    '4': 'intfloat/multilingual-e5-base', # Multilingual and strong performance
}

In [79]:
%%time
model = SentenceTransformer(models_available['4'])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10700.30it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [81]:
def generate_embeddings(chunks: list, model, batch_size: int = BATCH_SIZE) -> list:
    """Generate embeddings for text chunks using the model."""
    texts = [chunk["text"] for chunk in chunks]
    
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    return embeddings

In [98]:
# Generate embeddings for test chunks
embeddings_test = generate_embeddings(checking_chunks, model)
print(f"Embeddings shape: {embeddings_test.shape}")
print(f"Embedding dimension: {embeddings_test.shape[1]}")

Batches: 100%|██████████| 122/122 [00:14<00:00,  8.26it/s]


Embeddings shape: (3898, 768)
Embedding dimension: 768


## Creating FAISS Index

Building efficient vector index for similarity search

In [85]:
def create_faiss_index(embeddings):
    """Create FAISS index from embeddings."""
    embeddings = np.array(embeddings).astype('float32')
    
    # Create index using L2 distance
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    
    return index

In [99]:
# Create index for test embeddings
index_test = create_faiss_index(embeddings_test)
print(f"FAISS index created with {index_test.ntotal} vectors")

FAISS index created with 3898 vectors


## Retrieval: Search Function

Query embeddings and retrieve relevant documents

In [88]:
TOP_K = 5

In [89]:
def retrieve(query: str, index, chunks: list, model, top_k: int = TOP_K) -> list:
    """Retrieve top-k relevant chunks for a query."""
    # Encode query
    query_embedding = model.encode([query], convert_to_numpy=True).astype('float32')
    
    # Search in FAISS index
    distances, indices = index.search(query_embedding, top_k)
    
    # Retrieve relevant chunks
    results = []
    for i, idx in enumerate(indices[0]):
        if idx != -1:
            chunk = chunks[idx]
            results.append({
                "rank": i + 1,
                "distance": float(distances[0][i]),
                "doc_id": chunk["doc_id"],
                "chunk_id": chunk["chunk_id"],
                "filename": chunk["filename"],
                "text": chunk["text"][:200] + "..." if len(chunk["text"]) > 200 else chunk["text"]
            })
    
    return results

## Testing Retrieval

In [100]:
# Test with a sample PT-BR query
test_query = "Como calcular o imposto de renda?"
results = retrieve(test_query, index_test, checking_chunks, model, top_k=3)

print(f"Query: {test_query}\n")
for result in results:
    print(f"Rank {result['rank']}: {result['filename']} (Doc ID: {result['doc_id']}, Chunk: {result['chunk_id']})")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Text: {result['text']}\n")

Query: Como calcular o imposto de renda?

Rank 1: Declaração de Imposto de Renda Retido na Fonte (DIRF).txt (Doc ID: 71, Chunk: 4)
Distance: 0.2671
Text: de e-participação Termos de Uso Governo Digital Guia de Edição de Serviços do Portal Gov.br Canais do Executivo Federal Dados do Governo Federal Dados Abertos Painel Estatístico de Pessoal Painel de C...

Rank 2: Ato Declaratório Normativo Cosit nº 24, de 14 de setembro de 1999.txt (Doc ID: 23, Chunk: 20)
Distance: 0.2767
Text: , quando a apuração do imposto de rendafor com base no lucro real trimestral, com base no balanço e/oubalancete de redução e no lucro real apurado em 31 de dezembro doano-calendário (ajuste anual) ser...

Rank 3: Ato Declaratório Normativo CST nº 29, de 25 de junho de 1986.txt (Doc ID: 19, Chunk: 2)
Distance: 0.2852
Text: , como remuneração por serviços prestados sem vínculo empregatício com a fonte pagadora, não estão sujeitos à retenção do imposto de renda na fonte

